In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import spacy
import re
import contractions
from string import punctuation
from textblob import TextBlob
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_community.document_loaders import PyPDFLoader,TextLoader

In [27]:
loader = TextLoader('Data1.txt')
text = loader.load()
data = text[0].page_content

### TEXT NORMALIZATION

#### LOWERCASE

In [28]:
data = data.lower()

#### REMOVING EXTRA SPACES

In [29]:
data = re.sub(r'\s{2}','',data)

#### CONTRACTIONS

In [30]:
data = contractions.fix(data)

#### REMOVING SPECIAL CHARACTER

In [31]:
data = re.sub(r'[^0-9a-zA-Z]','',data)

#### TEXTBLOB

In [32]:
# values = TextBlob(data).correct()

#### SPACY LEMMATIZATION

In [33]:
nlp = spacy.load("en_core_web_sm")
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()

#### CHUNKING

In [34]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 250
)
chunks = splitter.create_documents([data])

#### EMBEDDINGS

In [35]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### VECTOR DB

In [36]:
vectordb = FAISS.from_documents(
    documents= chunks,
    embedding= embedding_model
)

In [37]:
user_query = "What is nlp?"
r_chunks = vectordb.similarity_search(user_query)

In [38]:
updated_r_chunks = set()
for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
r_text = '\n' . join (updated_r_chunks)    

#### RETRIVAL PART

In [39]:
def r_search(query,k=3): 
    r_chunks = vectordb.similarity_search(query,k=k)
    #print(R_chunks)
    r_chunks_chunks = ([doc.page_content for doc in r_chunks])
    r_Text = '\n'.join(r_chunks_chunks)
    
    # return(r_chunks)
def g_text(r_search,query):
    import os
    prompt= f''' You're an helpful assistant
                Assigned Task for you : Structure my Output =>{r_search}
                for this input=>{query}
                Note:
                1) Don't add extra contents just structure mentioned output
                2)If there is a mistake in output please correct or else please keep the original output
                  with structured result.
                  Output structure:
                  Input:{query}
                  Output:structure output'''

    llm_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # gemini-3.6-flash,gemini-2.5-pro
    api_key= os.environ["GEMINI_API_KEY"],
    temperature = 0.0
    ) 

    response = llm_model.invoke(prompt).content
    return response

user_prompt ='Explain Machine Learning ? '
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response = r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

[{'type': 'text', 'text': 'Input: Explain Machine Learning  \nOutput: \n\n### **What is Machine Learning?**\nMachine Learning (ML) is a branch of artificial intelligence (AI) that enables computers to learn from data and make decisions or predictions without being explicitly programmed.\n\n---\n\n### **How It Works**\n1. **Data Input:** Large amounts of data are fed into the system.\n2. **Pattern Recognition:** Algorithms analyze the data to find hidden patterns and relationships.\n3. **Model Training:** The system builds a mathematical model based on these patterns.\n4. **Prediction/Output:** The model uses what it learned to make predictions or decisions on new, unseen data.\n\n---\n\n### **Three Main Types of Machine Learning**\n\n*   **1. Supervised Learning**\n    *   *Concept:* Learning with a guide. The model is trained on labeled data (input-output pairs).\n    *   *Example:* Predicting house prices based on size and location, or classifying emails as "Spam" or "Not Spam".\n\n*

#### LCEL

In [40]:
parser = StrOutputParser()

In [ ]:
import os
llm = ChatGoogleGenerativeAI(
    model ='gemini-3.5-flash',
    api_key = os.environ["GEMINI_API_KEY"],
    temperature = 0.2
    
)


In [42]:
prompt = ChatPromptTemplate.from_messages([
    '''
    You are a helpful assistant.I assign a task for you.
    Check the retrived chunk Accuracy score.
    Example:
    Question => What is nlp?
    Retrived out of 100% chunks 80% chunks are correct.Return the score
    Query : {user_query}
    Retrival_chunk :{r_chunks}
     
    '''
]
)

In [43]:
chain = prompt | llm | parser

In [44]:
response = chain.invoke({"user_query":user_query,"r_chunks":r_chunks})

In [45]:
print(response)

Based on the query **"What is nlp?"**, here is the evaluation of the retrieved chunks:

1. **Chunk 1** (`id='ddffc7de...'`): **Correct (Relevant)**  
   * *Reasoning:* It directly defines NLP ("naturallanguageprocessingnlpisafascinatingfieldofartificialintelligence...") and explains its core purpose of enabling computers to understand human language.
2. **Chunk 2** (`id='5032ca26...'`): **Correct (Relevant)**  
   * *Reasoning:* It explains key components and preprocessing steps of NLP (tokenization, lowercasing, stopword removal), which are essential to understanding what NLP is and how it works.
3. **Chunk 3** (`id='675e0941...'`): **Correct (Relevant)**  
   * *Reasoning:* It continues explaining NLP preprocessing techniques (stemming, lemmatization) and introduces modern NLP applications (machine translation, sentiment analysis).
4. **Chunk 4** (`id='7cbf1587...'`): **Correct (Relevant)**  
   * *Reasoning:* It lists major NLP applications and modern architectures (chatbots, virtua